# `json2vec` Hello World

This notebook trains the smallest useful JSON2Vec model: two numeric Iris measurements predict the Iris species. The point is not accuracy. The point is to show the complete loop from records, to schema, to training, to prediction and embeddings. This example is intentionally flat; the next tutorials show why arrays matter.

Start with the normal training dependencies plus the bundled Iris JSONL buffer. The examples remove notebook logging noise so the rendered docs stay focused on model behavior.


In [1]:
import lightning.pytorch as lit
import polars as pl
import torch
from loguru import logger
from rich.pretty import pprint

import json2vec as j2v

logger.remove()

Load a tiny balanced slice of Iris rows. The schema field names match the DataFrame columns, so JSON2Vec can infer the request queries.


In [2]:
records = pl.read_ndjson("docs/data/iris.jsonl").head(36)

records.head()

sepal_length,sepal_width,petal_length,petal_width,species
f64,f64,f64,f64,str
5.1,3.5,1.4,0.2,"""setosa"""
7.0,3.2,4.7,1.4,"""versicolor"""
6.3,3.3,6.0,2.5,"""virginica"""
4.9,3.0,1.4,0.2,"""setosa"""
6.4,3.2,4.5,1.5,"""versicolor"""


The schema declares exactly what the model should read. `Number` fields become numeric tensorfields, and the `Category` field is a supervised target because `target=True` hides it from the input and asks the model to decode it.

In [3]:
model = j2v.Model.from_schema(
    j2v.Number("sepal_length"),
    j2v.Number("petal_length"),
    j2v.Category("species", target=True, max_vocab_size=4, topk=[2]),
    d_model=16,
    n_layers=1,
    n_heads=4,
    batch_size=8,
    embed=True,
    optimizer=lambda module: torch.optim.AdamW(module.parameters(), lr=1e-2),
)

datamodule = j2v.PolarsDataModule(
    model,
    train=records,
    validate=records,
    num_workers=0,
    persistent_workers=False,
    pin_memory=False,
    observation_buffer_size=32,
    sample_rate=1.0,
)

Train for one deliberately small epoch. The tutorials keep batch and epoch counts hardcoded so the example remains quick to run in documentation builds.

In [4]:
trainer = lit.Trainer(
    accelerator="cpu",
    max_epochs=1,
    logger=False,
    enable_progress_bar=False,
    enable_model_summary=False,
    enable_checkpointing=False,
    limit_train_batches=1,
    limit_val_batches=1,
)

trainer.fit(model=model, datamodule=datamodule)

GPU available: True (mps), used: False


TPU available: False, using: 0 TPU cores


/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


`Trainer(limit_train_batches=1)` was configured so 1 batch per epoch will be used.


`Trainer(limit_val_batches=1)` was configured so 1 batch will be used.


/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=13` in the `DataLoader` to improve performance.
/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=13` in the `DataLoader` to improve performance.
`Trainer.fit` stopped: `max_epochs=1` reached.


Prediction uses the same nested batch shape as training: each outer item is one observation, and each observation contains one record.

In [5]:
batch = [[record] for record in records.to_dicts()[:3]]

In [6]:
pprint(model.predict(batch))

{
│   'record/species': {
│   │   'state': {
│   │   │   'valued': [0.23748287558555603, 0.24646733701229095, 0.24384713172912598],
│   │   │   'null': [0.17446094751358032, 0.1797579675912857, 0.1784108281135559],
│   │   │   'padded': [0.29484260082244873, 0.29493898153305054, 0.29505351185798645],
│   │   │   'masked': [0.09862297028303146, 0.09333495795726776, 0.09642233699560165],
│   │   │   'other': [0.19459058344364166, 0.18550093472003937, 0.1862662136554718]
│   │   },
│   │   'content': {
│   │   │   'value': ['setosa', 'setosa', 'setosa'],
│   │   │   'probability': [0.7150182723999023, 0.6854256391525269, 0.6892114281654358],
│   │   │   'topk': [
│   │   │   │   [
│   │   │   │   │   {'label': 'setosa', 'probability': 0.7150182723999023},
│   │   │   │   │   {'label': 'virginica', 'probability': 0.18272122740745544}
│   │   │   │   ],
│   │   │   │   [
│   │   │   │   │   {'label': 'setosa', 'probability': 0.6854256391525269},
│   │   │   │   │   {'label': 'virginica', 'probability': 0.21159937977790833}
│   │   │   │   ],
│   │   │   │   [
│   │   │   │   │   {'label': 'setosa', 'probability': 0.6892114281654358},
│   │   │   │   │   {'label': 'virginica', 'probability': 0.20671214163303375}
│   │   │   │   ]
│   │   │   ]
│   │   }
│   }
}

Embeddings are opt-in. Passing `embed=True` when constructing the model exposes a root `record` vector for each input observation without changing the schema fields themselves.


In [7]:
pprint(model.embed(batch))

{
│   'record': {
│   │   'embedding': [
│   │   │   [
│   │   │   │   -0.11620007455348969,
│   │   │   │   0.07270009815692902,
│   │   │   │   -0.3406486511230469,
│   │   │   │   -0.10239608585834503,
│   │   │   │   0.049892671406269073,
│   │   │   │   0.5679291486740112,
│   │   │   │   -0.002381209982559085,
│   │   │   │   0.3576235771179199,
│   │   │   │   0.047423429787158966,
│   │   │   │   -0.09622805565595627,
│   │   │   │   -0.07797981053590775,
│   │   │   │   0.006487374193966389,
│   │   │   │   -0.3760238289833069,
│   │   │   │   -0.328191339969635,
│   │   │   │   0.3668253719806671,
│   │   │   │   -0.021367043256759644
│   │   │   ],
│   │   │   [
│   │   │   │   -0.19036923348903656,
│   │   │   │   0.04444563388824463,
│   │   │   │   -0.28295645117759705,
│   │   │   │   0.013194238767027855,
│   │   │   │   0.010814345441758633,
│   │   │   │   0.5300938487052917,
│   │   │   │   0.0037301178090274334,
│   │   │   │   0.3795350193977356,
│   │   │   │   -0.01373033132404089,
│   │   │   │   -0.12398587912321091,
│   │   │   │   -0.23239709436893463,
│   │   │   │   0.12588876485824585,
│   │   │   │   -0.3367711305618286,
│   │   │   │   -0.311321496963501,
│   │   │   │   0.4005953073501587,
│   │   │   │   -0.011778087355196476
│   │   │   ],
│   │   │   [
│   │   │   │   -0.15422146022319794,
│   │   │   │   0.03610830381512642,
│   │   │   │   -0.3028015196323395,
│   │   │   │   -0.004591938573867083,
│   │   │   │   0.027967015281319618,
│   │   │   │   0.5475949048995972,
│   │   │   │   0.032773055136203766,
│   │   │   │   0.3493841290473938,
│   │   │   │   0.018599556758999825,
│   │   │   │   -0.13598424196243286,
│   │   │   │   -0.17468486726284027,
│   │   │   │   0.1352691948413849,
│   │   │   │   -0.3476710021495819,
│   │   │   │   -0.34356075525283813,
│   │   │   │   0.38555410504341125,
│   │   │   │   -0.06484673172235489
│   │   │   ]
│   │   ]
│   }
}

The plot is the quickest way to verify what was built: array nodes, tensorfield nodes, targets, embeddings, and parameter counts all appear in the same tree.

In [8]:
model.plot()

Schema
record [array] embed
record
d_model=16  attention=mha  max_length=1  n_outputs=1  n_layers=1  n_heads=4  batch_size=8  parameters=18,153  arrays=1  
fields=3  targets=1  embeds=1  embed=True  n_linear=1
├── sepal_length [number] 3,943 params
│   record/sepal_length
│   query=[*].sepal_length  pooling=query  objective=mae  weight=1.0  n_heads=4  n_linear=1  jitter=0.0  n_bands=8  
│   offset=4
├── petal_length [number] 3,943 params
│   record/petal_length
│   query=[*].petal_length  pooling=query  objective=mae  weight=1.0  n_heads=4  n_linear=1  jitter=0.0  n_bands=8  
│   offset=4
└── species [category] target 3,659 params
    record/species
    query=[*].species  pooling=query  max_vocab_size=4  topk=[2]  weight=1.0  n_heads=4  p_prune=1.0  n_linear=1  
    n_bands=8  p_unavailable=0.01